- NAME          : ANISH SAMUEL
- COURSE        : DSC670-T301 Advanced Uses of Generative AI
- ASSIGNMENT    : WEEK 6

## Approach
- Prepare and upload training data - Load the IMDb dataset with `datasets` from hugging face
- Choose a base model -  `distilbert-base-uncased`
- Create the fine-tuning job - `Trainer.train()`
- Check the job's results - `Trainer.evaluate()`
- Run predictions on new reviews - deciding whether a movie review is positive or negative.

## 1. Setup

In [11]:
import os
import numpy as np
import torch

os.environ["WANDB_DISABLED"] = "true"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


## 2. Load the dataset
The current Hugging Face library for imdb is  `stanfordnlp/imdb`.

In [12]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")
print(dataset)
dataset["train"][0]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

## 3. Tokenize

We are using model `distilbert/distilbert-base-uncased` from hugging face.

In [ ]:
from transformers import AutoTokenizer

model_name = "distilbert/distilbert-base-uncased"

# Load the tokenizer for the model
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenize the dataset. The caseof text is converted to lowercase because the tokenizer is uncased. 
# The text is also truncated to a maximum length of 512 tokens, which is the maximum input length for DistilBERT.
def tokenize_function(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

# Tokenize the dataset using the defined function. 
# The batched=True argument allows for processing multiple examples at once, which can speed up the tokenization process.
tokenized_datasets = dataset.map(tokenize_function, batched=True)

print(tokenized_datasets["train"][0])

## 4. Load the model

In [ ]:
# Load the pre-trained model for sequence classification. 
# The num_labels parameter is set to 2 because the IMDB dataset has two classes: positive and negative reviews.
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

## 5. Train
Like the article, I train on a 2,000-review sample for 2 epochs and evaluate on 500 test reviews, which keeps training to a few minutes on a Colab GPU. I added an accuracy metric because loss alone doesn't show how often the model is right.

In [ ]:
from transformers import TrainingArguments, Trainer

# Define a function to compute evaluation metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {"accuracy": (np.argmax(logits, axis=-1) == labels).mean()}

# Prepare the training and evaluation datasets
train_data = tokenized_datasets["train"].shuffle(seed=42).select(range(2000))
eval_data = tokenized_datasets["test"].shuffle(seed=42).select(range(500))

# Define the training arguments. 
# These include the output directory for saving model checkpoints, evaluation strategy, batch sizes, 
# number of training epochs, weight decay for regularization, and reporting settings.
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    report_to="none",
)

# Create a Trainer instance
# Trainer is a high-level API provided by Hugging Face Transformers that simplifies the training and evaluation of models.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    compute_metrics=compute_metrics,
)

# Start the training process
# trainer.train() will begin the training of the model using the specified training arguments, datasets, and evaluation metrics.
trainer.train()

## 6. Evaluate

In [ ]:
# Evaluate the trained model
# evaluate() will compute the evaluation metrics on the evaluation dataset and return the results.
trainer.evaluate()

- Training loss of 0.30 shows it fit the training reviews well.
- Validation loss of 0.44 shows it does somewhat worse on new reviews. That gap is mild overfitting, which is expected when training on only 2,000 examples.

## 7. Test the model on new reviews

In [ ]:
labels = {0: "NEGATIVE", 1: "POSITIVE"}
# Move the model to the specified device
model.to(device)

# eval() sets the model to evaluation mode, which is necessary for inference.
model.eval()

# Define a list of test reviews to evaluate the model's performance on unseen data.
test_reviews = [
    "I absolutely loved this movie. The story was gripping, and the performances were phenomenal!",
    "This was a terrible movie. The plot made no sense and the acting was awful.",
    "The movie was not bad at all.",
    "Great actors, but the script was so boring I fell asleep.",
    "Oh wonderful, another two hours of my life I will never get back.",
    "I slept through the entire movie."
]

# Evaluate the model on the test reviews
for text in test_reviews:
    inputs = tokenizer(text, return_tensors="pt", truncation=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)[0]
    pred = probs.argmax().item()
    print(f"{labels[pred]:8s} ({probs[pred]:.2f})  {text}")

## Conclusion
The model got 4 of the 6 test reviews right.
 - The movie was not bad at all. - Should have been positive
 - Oh wonderful, another two hours of my life I will never get back. - Should have been negative

Clear reviews are easy. Reviews with obvious words like "loved", "terrible" and "awful" are classified correctly.
Mixed opinions worked. In "Great actors, but… boring," the model correctly weighted the part after "but" over the earlier praise.
It could not spot sarcasm.
Both mistakes were made at 99% confidence, so a high score doesn’t mean the answer is right. That’s a risk if the model were used without a person checking it.
